Parameter optimization
================

This notebook shows how both hyperparameters and data source weights can be optimized. This code merely serves as a demonstration of the process. 

In [1]:
from nichenetpy.utils import (
    read_csv_cols
)
from nichenetpy.parameter_optimization import (
    construct_and_evaluate
)

from optuna import create_study
from optuna.trial import Trial
from optuna.samplers import TPESampler
from itertools import chain

import os
import requests
import pandas as pd
import session_info
import json
import numpy as np

c:\Users\victorm\Documents\nichenetpy\.hatch\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
network_path = os.path.normpath("./tutorial_files/model_construction/human")
if not os.path.exists(network_path):
    os.makedirs(network_path)
for filename in (
    "gr_human.csv",
    "lr_network_human.csv",
    "lr_sig_human.csv",
    "optimized_source_weights.csv",
    "annotation_data_sources.csv"
):
    file_path = os.path.join(network_path, filename)
    if not os.path.exists(file_path):
        res = requests.get(f"https://zenodo.org/records/14929618/files/{filename}")
        with open(file_path, "wb") as file:
            file.write(res.content)
train_path = os.path.normpath("./tutorial_files/model_optimization")
if not os.path.exists(train_path):
    os.makedirs(train_path)
for filename in (
    "settings_trainingf1234.json",
    "settings_trainingf1235.json",
    "settings_trainingf1245.json",
    "settings_trainingf1345.json",
    "settings_trainingf2345.json"
):
    file_path = os.path.join(train_path, filename)
    if not os.path.exists(file_path):
        res = requests.get(f"https://zenodo.org/records/15799578/files/{filename}")
        with open(file_path, "wb") as file:
            file.write(res.content)

In [3]:
gr_network = pd.DataFrame(read_csv_cols(os.path.join(network_path, "gr_human.csv")))
lr_network = pd.DataFrame(read_csv_cols(os.path.join(network_path, "lr_network_human.csv")))
sig_network = pd.DataFrame(read_csv_cols(os.path.join(network_path, "lr_sig_human.csv")))

In [4]:
train_path = "D:/Data/nichenetpy/model_optimization"
with open(os.path.join(train_path, "settings_training_f1234.json"), "rb") as file:
    settings_CV = json.loads(file.read())
settings = settings_CV["settings"]

In [5]:
gr_network = gr_network[
    ((gr_network["database"] == "NicheNet_LT") & np.array([fr not in settings_CV["forbidden_ligands_nichenet"] for fr in gr_network["from"]]))
    |
    ((gr_network["database"] == "CytoSig") & np.array([fr not in settings_CV["forbidden_ligands_cytosig"] for fr in gr_network["from"]]))
]

In [ ]:
source_names = sorted(set(chain(gr_network["source"], lr_network["source"], sig_network["source"])))

def objective(trial:Trial):
    source_weights = dict(
        (
            source_name,
            trial.suggest_float(
                name=source_name,
                low=0,
                high=1
            )
        ) for source_name in source_names
    )
    lr_sig_hub = trial.suggest_float(
        name="lr_sig_hub",
        low=0,
        high=1
    )
    gr_hub = trial.suggest_float(
        name="gr_hub",
        low=0,
        high=1
    )
    ltf_cutoff = trial.suggest_float(
        name="ltf_cutoff",
        low=0.9,
        high=0.999
    )
    damping_factor = trial.suggest_float(
        name="damping_factor",
        low=0.01,
        high=0.99
    )
    res = construct_and_evaluate(
        source_weights,
        lr_sig_hub,
        gr_hub,
        ltf_cutoff,
        damping_factor,
        lr_network,
        gr_network,
        sig_network,
        settings
    )
    return (res[1], res[2])

study = create_study(
    sampler=TPESampler(),
    directions=["maximize", "maximize"]
)
study.optimize(
    objective,
    n_trials=3,
    n_jobs=-1
)

[I 2025-07-03 15:44:13,940] A new study created in memory with name: no-name-c950687c-6189-4bc1-9063-70ab95e448a7


In [ ]:
best_params = [trial.params for trial in study.best_trials]
len(best_params)

3

In [ ]:
session_info.show()